# Experiment: AKI Data Cleaning

Objective:
- Load `mimic_aki_cohort_raw.csv` exported from BigQuery.
- Review missingness, separate identifiers / timing / target columns, and remove obviously unusable columns.
- Preserve `subject_id` for later subject-level model splitting.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

INPUT_CSV = Path('outputs/mimic_aki_cohort_raw.csv')
OUTPUT_DIR = Path('outputs')
CLEANED_CSV = OUTPUT_DIR / 'mimic_aki_cohort_cleaned.csv'
DATA_DICTIONARY_CSV = OUTPUT_DIR / 'mimic_aki_data_dictionary.csv'
TARGET_COL = 'future_aki_24h'
ID_COLS = ['subject_id', 'hadm_id', 'stay_id']
TIME_COLS = [
    'icu_intime',
    'anchor_time',
    'obs_window_start',
    'obs_window_end',
    'pred_window_start',
    'pred_window_end',
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'input_csv': str(INPUT_CSV), 'cleaned_csv': str(CLEANED_CSV)})


## Load raw dataset

This notebook is intended for Google Colab after `data_fetch_aki.py` has exported the raw AKI cohort CSV.


In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Raw shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print('Target prevalence:')
print(df[TARGET_COL].value_counts(dropna=False).sort_index())
df.head()


## Basic inspection

Start with a compact summary of column types, unique counts, and missingness.


In [ ]:
column_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(dropna=False),
    'missing_count': df.isna().sum(),
})
column_summary['missing_pct'] = (column_summary['missing_count'] / len(df) * 100).round(2)
column_summary.sort_values(['missing_pct', 'n_unique'], ascending=[False, True]).head(20)


## Missingness review

High-missingness columns are not automatically leakage, but they are good candidates for pruning before baseline models.


In [ ]:
missingness = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename('missing_pct')
      .reset_index()
      .rename(columns={'index': 'column'})
)
missingness.head(30)


## Column grouping

Separate identifiers, timing columns, the target, and candidate feature columns. Keep `subject_id` so model splitting can remain subject-level.


In [ ]:
missing_threshold = 95.0
required_keep_cols = set(ID_COLS + TIME_COLS + [TARGET_COL])

high_missing_cols = [
    col for col, pct in missingness.set_index('column')['missing_pct'].items()
    if pct >= missing_threshold and col not in required_keep_cols
]
constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1 and col not in required_keep_cols
]
explicit_leakage_cols = [
    col for col in [
        'icu_outtime',
        'dischtime',
        'los_icu',
        'los_hospital',
        'hospital_expire_flag',
        'aki_onset_time',
        'aki_onset_stage',
        'aki_stage_max',
        'has_aki_anytime',
        'eligible_for_prediction',
    ]
    if col in df.columns
]

feature_candidate_cols = [
    col for col in df.columns
    if col not in required_keep_cols and col not in explicit_leakage_cols
]

print('ID columns:', ID_COLS)
print('Time columns:', TIME_COLS)
print('Target column:', TARGET_COL)
print(f'Feature candidate count: {len(feature_candidate_cols)}')
print('High missing columns:', high_missing_cols)
print('Constant columns:', constant_cols)
print('Explicit leakage review columns:', explicit_leakage_cols)


## Cleaning decisions

This first-pass cleaned file removes obviously unusable columns while preserving identifiers, timing metadata, and the target. Modeling notebooks can further exclude timing columns from features without losing auditability.


In [ ]:
drop_cols = sorted(set(high_missing_cols + constant_cols + explicit_leakage_cols) - {'subject_id'})
cleaned_df = df.drop(columns=drop_cols, errors='ignore').copy()

for col in TIME_COLS:
    if col in cleaned_df.columns:
        cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

cleaned_df = cleaned_df.sort_values(['subject_id', 'stay_id']).reset_index(drop=True)
print(f'Cleaned shape: {cleaned_df.shape[0]:,} rows x {cleaned_df.shape[1]:,} columns')
print('Dropped columns:', drop_cols)
cleaned_df.head()


## Export cleaned dataset

The cleaned CSV remains row-level at `stay_id` grain and keeps `subject_id` for subject-level splitting in the modeling notebook.


In [ ]:
cleaned_df.to_csv(CLEANED_CSV, index=False)

data_dictionary = pd.DataFrame({
    'column': cleaned_df.columns,
    'dtype': cleaned_df.dtypes.astype(str).values,
    'missing_pct': cleaned_df.isna().mean().mul(100).round(2).values,
})
data_dictionary.to_csv(DATA_DICTIONARY_CSV, index=False)

print(f'Saved cleaned dataset to {CLEANED_CSV}')
print(f'Saved data dictionary to {DATA_DICTIONARY_CSV}')


## Next steps

- Review the dropped-column list before training.
- Keep subject-level splits by `subject_id` in downstream modeling.
- If the first Colab run reveals unexpected sparsity, adjust thresholds here rather than redefining the target.
